# §F — Served LLM+NTK NLL gate

Release-gate check for **NTK Phase-2**: confirm the *served* `ntk_model` ISVC reproduces the controller's hook-attached reference NLL.

It hits a deployed KServe ISVC's OpenAI `/v1/completions` (`echo` + `logprobs=1`) and computes the **same** teacher-forced, completion-masked NLL as the §E `evaluate_nll.py` `_completion_nll` arm, so the served number is apples-to-apples with the (reused) hook-attached `reference`.

**Apples-to-apples by construction:** we tokenize `prompt`+`completion` locally with the SAME base tokenizer and SAME split as the reference (prompt with special tokens; completion with `add_special_tokens=False`), then send the **exact token ids** as the completions `prompt`. vLLM uses them verbatim → no server-side retokenization drift; the completion span is exactly `token_logprobs[len(prompt_ids):]`.

**Gate:** `|served − reference| / reference ≤ threshold` (default 1%). Served ≈ reference is expected — the `NTKQwen2ForCausalLM` plugin attaches the controller with the same `ControllerRuntime.apply` hooks the reference used.

Prereqs: a deployed `ntk_model` ISVC (deploy **without** `--quantization` for an apples-to-apples comparison); the `reference` NLL computed on the **same** controller now being served; the GSM8K `eval.jsonl`.


In [ ]:
# !pip install transformers requests

In [ ]:
# --- Config -------------------------------------------------------------
BASE = "Qwen/Qwen2.5-0.5B-Instruct"          # same base the ISVC serves
EVAL_JSONL = "runs/gsm8k_small/eval.jsonl"   # GSM8K eval split (prepare_dataset.py)
ENDPOINT = "https://<isvc-host>/openai/v1"   # OpenAI base; POSTs to <endpoint>/completions
SERVED_MODEL_NAME = "<served_model_name>"    # vLLM --model_name / the "model" field
API_KEY = None                               # Bearer token if the gateway needs one
REFERENCE_NLL = 0.5888                        # hook-attached reference for the SAME controller
THRESHOLD = 0.01                              # |served-reference|/reference pass band
MAX_TOKENS = 0                                # 0 = echo only; auto-falls back to 1 if rejected

In [ ]:
import json

def load_examples(path):
    """Read the {prompt, completion} JSONL (same contract as the trainer / §E harness)."""
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def completion_token_split(tokenizer, prompt, completion):
    """Reproduce _completion_nll's split exactly: prompt with special tokens,
    completion with add_special_tokens=False, concatenated. Returns int lists."""
    prompt_ids = tokenizer(prompt).input_ids
    completion_ids = tokenizer(completion, add_special_tokens=False).input_ids
    return prompt_ids, completion_ids, prompt_ids + completion_ids

In [ ]:
def completion_logprobs_from_echo(logprobs_obj, input_ids, completion_start):
    """Slice per-token logprobs for the completion span from an echo response.

    Guards (Copilot review):
      * logprobs_obj must be a dict — a server returning ``logprobs: null``
        would otherwise raise an opaque AttributeError.
      * a None *inside* the completion span is a hard error — never coerce to
        0.0 (that would understate NLL -> false PASS).
    """
    if not isinstance(logprobs_obj, dict):
        raise RuntimeError(
            f"served response 'logprobs' is missing or not an object "
            f"(got {type(logprobs_obj).__name__}); the server returned no logprobs"
        )
    token_logprobs = logprobs_obj.get("token_logprobs")
    if token_logprobs is None:
        raise RuntimeError("served response has no logprobs.token_logprobs")
    if len(token_logprobs) < len(input_ids):
        raise RuntimeError(
            f"served logprobs length {len(token_logprobs)} < input length "
            f"{len(input_ids)}; cannot align completion span"
        )
    comp = token_logprobs[completion_start:len(input_ids)]
    if any(lp is None for lp in comp):
        raise RuntimeError(
            "served logprobs contain None inside the completion span; "
            "the server did not return a logprob for every completion token"
        )
    return comp

In [ ]:
import requests

def _post_completion(url, headers, base_body, max_tokens):
    return requests.post(url, headers=headers, json={**base_body, "max_tokens": max_tokens}, timeout=120)

def served_completion_nll(endpoint, served_model_name, api_key, tokenizer, examples, max_tokens=0):
    """Teacher-forced completion-NLL via the served OpenAI endpoint.

    Same aggregation as _completion_nll: global sum of per-completion-token
    negative logprob / total completion tokens (NOT mean-of-means).

    max_tokens fallback (Copilot review): some vLLM builds reject max_tokens=0
    with echo; on a 400 we retry once with max_tokens=1 (the trailing generated
    token is sliced off) and use 1 for the rest of the run.
    """
    url = endpoint.rstrip("/") + "/completions"
    headers = {"Content-Type": "application/json"}
    if api_key:
        headers["Authorization"] = f"Bearer {api_key}"

    eff = max_tokens
    total_neg_logprob, total_tokens = 0.0, 0
    for i, ex in enumerate(examples):
        prompt_ids, completion_ids, input_ids = completion_token_split(
            tokenizer, ex["prompt"], ex["completion"])
        if not completion_ids:
            continue
        base_body = {
            "model": served_model_name,
            "prompt": input_ids,   # token ids -> used verbatim, no retokenization
            "echo": True,
            "logprobs": 1,
            "temperature": 0,
        }
        resp = _post_completion(url, headers, base_body, eff)
        if resp.status_code == 400 and eff == 0:
            eff = 1  # server rejects max_tokens=0; fall back for this + remaining
            resp = _post_completion(url, headers, base_body, eff)
        resp.raise_for_status()
        choice = resp.json()["choices"][0]
        comp = completion_logprobs_from_echo(choice["logprobs"], input_ids, len(prompt_ids))
        total_neg_logprob += -float(sum(comp))
        total_tokens += len(comp)
        if (i + 1) % 10 == 0:
            print(f"[served] {i + 1} examples scored")

    if total_tokens == 0:
        raise RuntimeError("no completion tokens scored")
    return total_neg_logprob / total_tokens

In [ ]:
import math
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE)
examples = load_examples(EVAL_JSONL)
print(f"scoring {len(examples)} examples against {ENDPOINT}")

served_nll = served_completion_nll(
    ENDPOINT, SERVED_MODEL_NAME, API_KEY, tokenizer, examples, max_tokens=MAX_TOKENS)

print("\n================== §F served-arm summary ==================")
print(f"served (LLM+NTK ISVC)       : {served_nll:.4f}  ({math.exp(served_nll):.4f} ppl)")
rel = abs(served_nll - REFERENCE_NLL) / max(REFERENCE_NLL, 1e-9)
verdict = "PASS" if rel <= THRESHOLD else "FAIL"
print(f"reference (reused §E hooks)  : {REFERENCE_NLL:.4f}  "
      f"(Δ = {served_nll - REFERENCE_NLL:+.4f}, relative = {rel*100:.2f}%, "
      f"threshold {THRESHOLD*100:.0f}%, {verdict})")

## Notes

- **Reference must match the served controller.** `REFERENCE_NLL` is only valid if it was computed (via the §E `evaluate_nll.py` `reference` arm) on the *exact* `controller.pt` this ISVC serves.
- **No quantization on the gate ISVC** — deploy without `--quantization` so served (bf16/fp16) vs reference (fp32) differ only by float noise, which the 1% band absorbs. A bigger gap is a serving-path bug to investigate, not a band to widen.
- **Spot-check first:** on one example, the negated per-token logprobs here should match the reference arm's per-token cross-entropy within fp tolerance before trusting the aggregate.
